<a href="https://colab.research.google.com/github/JasonL888/AI_Experiments/blob/main/RAG_Agent/rag_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Custom RAG



## Pre-requisites
- Create HuggingFace read-only token
- if running locally
  - update `.env` with `HF_TOKEN=<token-value>`
    - where `<token-value> is your HuggingFace token
  - upload your PDFs to `./pdfs`

- if running on colab
  - add to Secrets `HF_TOKEN=<token-value>`
    - where `<token-value> is your HuggingFace token
    - toggle to enable `Notebook access`
  - when prompted upload the PDF files

### Install and import pre-requisite packages

In [1]:
# only local env
# - if local, uncomment
# - not needed for Colab
#!pip install torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 transformers sentence-transformers

In [1]:
!pip install chromadb pypdf

In [2]:
!pip install bitsandbytes accelerate

In [3]:
import os
import glob
import sys
from getpass import getpass

from typing import List, Dict, Tuple
from dotenv import load_dotenv

import pypdf
import chromadb
from chromadb.utils import embedding_functions

from transformers import pipeline # For LLM pipeline

IN_COLAB = "google.colab" in sys.modules

### Handle HuggingFace Token

In [4]:
# Handle the HuggingFace Token
# HF_TOKEN handling: prefer environment or .env, otherwise prompt
load_dotenv()
hf = os.environ.get("HF_TOKEN")

if not hf:
    # Colab: ask for HF token (hidden) and persist to .env
    if IN_COLAB:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
        if token:
            os.environ["HF_TOKEN"] = token
            # persist token for future cells (writes .env in workspace)
            with open(".env", "a") as envf:
                envf.write(f"\nHF_TOKEN={token}\n")
            print("HF_TOKEN set for session and appended to .env")
    else:
        # Local (VS Code): missing so prompt for it
        print("HF_TOKEN not found. Create a .env file with HF_TOKEN=your_token or paste it now.")
        token = getpass("Paste HF_TOKEN (hidden): ")
        if token:
            os.environ["HF_TOKEN"] = token
            with open(".env", "a") as envf:
                envf.write(f"\nHF_TOKEN={token}\n")
            print("HF_TOKEN written to .env")

### Upload PDFs

In [5]:

# Handle the PDFs for grounding RAG
# ensure pdf folder exists
PDF_DIR = "pdfs"
os.makedirs(PDF_DIR, exist_ok=True)
print(f"PDF directory: {os.path.abspath(PDF_DIR)}")

if IN_COLAB:
    # Colab: prompt for file upload (files.upload returns dict of filename->bytes)
    from google.colab import files
    print("Detected Google Colab. Use the file picker to upload your PDFs.")
    uploaded = files.upload()
    for filename, content in uploaded.items():
        dest = os.path.join(PDF_DIR, filename)
        with open(dest, "wb") as f:
            f.write(content)
        print(f"Saved uploaded file to: {dest}")



PDF directory: /content/pdfs
Detected Google Colab. Use the file picker to upload your PDFs.


## Configuration

In [6]:
# --- Configuration ---
load_dotenv()

# Model for creating embeddings (converts text into vectors).
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

# Local LLM model for the RAG generation phase.
# Trying a DeepSeek quantized instruction model
# Requires sufficient resources (Colab Pro/Pro+ likely needed)
FLAN_T5_MODEL = "deepseek-ai/DeepSeek-Coder-6.7B-Instruct-GPTQ"

# Name of the Chroma collection (the vector store database).
COLLECTION_NAME = "pdf_rag_collection"
CHROMA_DB_PATH = "./chroma_db"

# Text splitting parameters
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 600 # Further increased overlap

## Helper Functions

In [7]:
def split_text_to_chunks(text: str, source_path: str, page_num: int) -> List[Tuple[str, Dict]]:
    """
    A simple text splitter function to break down large documents.
    This mimics recursive splitting logic using simple overlap.
    Returns a list of (text_chunk, metadata) tuples.
    """
    chunks = []
    i = 0
    while i < len(text):
        end = i + CHUNK_SIZE
        chunk = text[i:end]

        # Add chunk and metadata
        metadata = {
            "source": source_path,
            "page": page_num,
            "chunk_index": len(chunks)
        }
        chunks.append((chunk, metadata))

        # Determine the next starting point
        i += CHUNK_SIZE - CHUNK_OVERLAP

    return chunks

In [8]:
from transformers import pipeline, BitsAndBytesConfig # For LLM pipeline and quantization config
import torch # Import torch for device handling

def initialize_llm_pipeline():
    """
    Initializes and returns the HuggingFace LLM pipeline using the specified model,
    attempting 8-bit quantization if possible.
    """
    try:
        print(f"\n--- Initializing HuggingFace Model: {FLAN_T5_MODEL} ---")

        # Configure quantization
        # Attempt 8-bit quantization
        quantization_config = BitsAndBytesConfig(
            load_in_8bit=True,
        )

        # Load model with quantization config
        # Remove device argument when using quantization with accelerate
        llm_pipeline = pipeline(
            "text2text-generation", # Still using text2text for T5 models
            model=FLAN_T5_MODEL,
            model_kwargs={"quantization_config": quantization_config}
        )
        print("Model loaded successfully (attempted 8-bit quantization).")
        return llm_pipeline
    except ImportError:
        print("\nError: Required libraries ('transformers', 'torch', 'bitsandbytes', or 'accelerate') not found.")
        print("Please ensure all prerequisites are installed.")
        return None
    except Exception as e:
        print(f"Error loading model {FLAN_T5_MODEL} with quantization: {e}")
        print("Falling back to loading without quantization...")
        # Fallback to loading without quantization if 8-bit fails
        try:
            # Determine device for fallback
            device = "cuda" if torch.cuda.is_available() else "cpu"
            print(f"Using device for fallback: {device}")
            llm_pipeline = pipeline(
                "text2text-generation",
                model=FLAN_T5_MODEL,
                device=device
            )
            print("Model loaded successfully (without quantization).")
            return llm_pipeline
        except Exception as fallback_e:
            print(f"Error loading model {FLAN_T5_MODEL} even without quantization: {fallback_e}")
            return None

In [9]:
def load_and_index_pdfs(directory_path: str) -> chromadb.Collection:
    """
    Loads all PDF files, chunks them page by page with overlap, and indexes them into a Chroma collection.

    Returns:
        A configured Chroma collection object.
    """
    print(f"--- Step 1 & 2: Loading and Chunking Documents from '{directory_path}' ---")

    if not os.path.isdir(directory_path):
        print(f"Error: Directory '{directory_path}' not found. Please create it.")
        return None

    pdf_files = glob.glob(os.path.join(directory_path, "*.pdf"))
    if not pdf_files:
        print(f"Warning: No PDF files found in '{directory_path}'.")
        return None

    all_chunks: List[Tuple[str, Dict]] = []
    total_pages = 0

    for file_path in pdf_files:
        try:
            reader = pypdf.PdfReader(file_path)
            num_pages = len(reader.pages)
            total_pages += num_pages

            for i in range(num_pages):
                page_text = reader.pages[i].extract_text() or ""

                # Create a chunk for the current page
                chunk_text = page_text

                # Add overlap with the next page if it exists
                if i + 1 < num_pages:
                    next_page_text = reader.pages[i+1].extract_text() or ""
                    # Take the first CHUNK_OVERLAP characters from the next page
                    overlap_text = next_page_text[:CHUNK_OVERLAP]
                    chunk_text += "\n" + overlap_text # Append overlap text

                # Add chunk and metadata
                metadata = {
                    "source": file_path,
                    "page": i, # 0-indexed page number
                    "chunk_index": len(all_chunks) # Index within the list of all chunks
                }
                all_chunks.append((chunk_text, metadata))

            print(f"Processed: {os.path.basename(file_path)} ({num_pages} pages)")

        except Exception as e:
            print(f"Could not process {file_path}: {e}")

    if not all_chunks:
        print("No readable content found. Exiting indexing.")
        return None

    print(f"Total pages processed: {total_pages}. Created {len(all_chunks)} text chunks.")

    # --- Step 3: Initializing Embeddings and Vector Store (Chromadb Native) ---
    print(f"\n--- Step 3: Creating Embeddings and Indexing (Model: {EMBEDDING_MODEL}) ---")

    try:
        # Initialize Chroma Client
        client = chromadb.PersistentClient(path=CHROMA_DB_PATH)

        # Initialize Sentence Transformer for embedding generation
        embedding_function = embedding_functions.SentenceTransformerEmbeddingFunction(
            model_name=EMBEDDING_MODEL,
            device='cpu' # Use CPU for local environments
        )

        # Create or retrieve the collection
        collection = client.get_or_create_collection(
            name=COLLECTION_NAME,
            embedding_function=embedding_function
        )

        # Prepare data for native Chroma addition
        texts = [chunk[0] for chunk in all_chunks]
        metadatas = [chunk[1] for chunk in all_chunks]
        ids = [f"doc_{i}" for i in range(len(texts))]

        # Clear existing data if necessary (optional)
        existing_ids = collection.get().get('ids', [])
        if existing_ids:
            collection.delete(ids=existing_ids)
        else:
            print("No existing documents to delete from collection.")

        # Add data to the collection
        collection.add(
            documents=texts,
            metadatas=metadatas,
            ids=ids
        )

        print(f"Indexing complete. Total documents in collection: {collection.count()}")
        return collection

    except Exception as e:
        print(f"An error occurred during embedding or vector store creation: {e}")
        return None

In [10]:
def invoke_llm_with_context(llm_pipeline, context: str, query: str) -> str:
    """
    Uses a HuggingFace pipeline to generate a response based on the context.

    (The content of this function remains largely the same, only the input
     source for the context changes from LangChain Document to raw string.)
    """
    if llm_pipeline is None:
        return "LLM pipeline is not initialized. Cannot generate response."

    print("\n--- LLM Invocation (HuggingFace Pipeline) ---")

    # Construct the RAG prompt template for the LLM
    # Refined prompt to be more direct and emphasize using context
    system_prompt = (
        "You are an intelligent assistant. Extract the answer to the following question *only* from the provided context. "
        "If the answer is not explicitly stated in the context, respond with 'The information is not available in the provided documents.' "
        "Provide a concise and detailed answer if the information is available."
    )
    prompt = f"{system_prompt}\n\nContext: {context}\n\nQuestion: {query}\n\nAnswer:"

    # Generate the response
    try:
        result = llm_pipeline(prompt, max_new_tokens=512, clean_up_tokenization_spaces=True) # Increased max_new_tokens
        generated_text = result[0]['generated_text']

        return generated_text.strip()

    except Exception as e:
        return f"Error during LLM generation: {e}"

In [11]:
def rag_query(vectorstore_collection: chromadb.Collection, prompt: str, llm_pipeline):
    """
    Performs the RAG retrieval and LLM generation using the native Chroma collection.

    Args:
        vectorstore_collection: The configured Chroma collection object.
        prompt: The user's question.
        llm_pipeline: The initialized HuggingFace pipeline object.
    """
    # --- Step 4: Retrieval (Chromadb Native) ---
    print(f"\n--- Step 4: Retrieval for Query: '{prompt}' ---")

    # Perform the semantic search
    results = vectorstore_collection.query(
        query_texts=[prompt],
        n_results=5, # Retrieve top 5 documents/chunks
        include=['documents', 'metadatas'] # Ask for the content and source info
    )

    # Extract documents (context) and metadata
    retrieved_documents = results.get('documents', [[]])[0]
    retrieved_metadatas = results.get('metadatas', [[]])[0]

    # Combine the content of the retrieved documents into a single context string with source information
    context_parts = []
    print("--- Sources ---")
    for i, (doc, metadata) in enumerate(zip(retrieved_documents, retrieved_metadatas)):
        source = metadata.get('source', 'Unknown Source')
        page = metadata.get('page', 0) + 1 # 0-indexed page number, display as 1-indexed
        context_parts.append(f"--- Document {i+1} (Source: {os.path.basename(source)}, Page: {page}) ---\n{doc}")
        print(f"Chunk {i+1}: Source: {os.path.basename(source)}, Page: {page}")

    context = "\n\n".join(context_parts)


    print(f"Found {len(retrieved_documents)} relevant chunks.")


    # Print the retrieved chunks for inspection (optional, can be commented out later)
    # print("\n--- Retrieved Chunks Content (Formatted for LLM) ---")
    # print(context)
    # print("--- End of Formatted Context ---")


    # --- Step 5: Generation ---
    final_response = invoke_llm_with_context(llm_pipeline, context, prompt)

    print("\n" + "="*50)
    print("FINAL RAG RESULT:")
    print(final_response)
    print("="*50)

## Load, Chunk and Index

In [12]:
vector_db = load_and_index_pdfs(PDF_DIR)
if vector_db:
  print("\nSuccessfully loaded and index the PDFs\n")
else:
  print("\nCould not initialize the vector store. Please check the directory and PDF files.")

--- Step 1 & 2: Loading and Chunking Documents from 'pdfs' ---
Processed: ABC_Bank_FAQ.pdf (21 pages)
Total pages processed: 21. Created 21 text chunks.

--- Step 3: Creating Embeddings and Indexing (Model: sentence-transformers/all-MiniLM-L6-v2) ---
Indexing complete. Total documents in collection: 21

Successfully loaded and index the PDFs



## Initialize LLM Pipeline

In [13]:
llm = initialize_llm_pipeline()
if llm:
  print("\nSuccessfully loaded the LLM pipeline\n")
else:
  print("\nCould not initialize the LLM pipeline. Please check the model name.")


--- Initializing HuggingFace Model: deepseek-ai/DeepSeek-Coder-6.7B-Instruct-GPTQ ---
Error loading model deepseek-ai/DeepSeek-Coder-6.7B-Instruct-GPTQ with quantization: deepseek-ai/DeepSeek-Coder-6.7B-Instruct-GPTQ is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `hf auth login` or by passing `token=<your_token>`
Falling back to loading without quantization...
Using device for fallback: cuda
Error loading model deepseek-ai/DeepSeek-Coder-6.7B-Instruct-GPTQ even without quantization: deepseek-ai/DeepSeek-Coder-6.7B-Instruct-GPTQ is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `hf auth login` or by passing `token=<your_token>`

Could not initialize the LLM

## Query LLM with RAG

In [14]:
user_query = input("\nEnter your query based on the PDF content (e.g., 'What is the minimum balance for a savings account ?'):\n> ")

if user_query:
    rag_query(vector_db, user_query, llm)
else:
    print("Query skipped.")


Enter your query based on the PDF content (e.g., 'What is the minimum balance for a savings account ?'):
> What is the minimum balance for a savings account ?

--- Step 4: Retrieval for Query: 'What is the minimum balance for a savings account ?' ---


Token indices sequence length is longer than the specified maximum sequence length for this model (3063 > 512). Running this sequence through the model will result in indexing errors


--- Sources ---
Chunk 1: Source: ABC_Bank_FAQ.pdf, Page: 15
Chunk 2: Source: ABC_Bank_FAQ.pdf, Page: 3
Chunk 3: Source: ABC_Bank_FAQ.pdf, Page: 9
Chunk 4: Source: ABC_Bank_FAQ.pdf, Page: 11
Chunk 5: Source: ABC_Bank_FAQ.pdf, Page: 7
Found 5 relevant chunks.

--- LLM Invocation (HuggingFace Pipeline) ---


/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:181: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")



FINAL RAG RESULT:
Savings account interest rates vary based on the type of account and current market conditions. Standard savings accounts offer a basic interest rate, while high-yield options provide higher returns. Rates are subject to change, so we recommend checking our website or contacting customer support for the latest information. Interest compounds monthly, helping your balance grow over time. Speak with a representative to determine the best savings option for your needs.
